# Demo Klasifikasi Email: Ham, Phish, atau Spam

Notebook ini memakai model akhir proyek untuk mengklasifikasikan email yang ditempel pengguna. Model akhir adalah *stacking* Multinomial Naive Bayes, Logistic Regression, dan LightGBM pada teks V3, dengan threshold Ham b. Konfigurasinya dipilih pada Pertemuan 7 dan diuji pada Evaluasi Akhir.

Notebook ini menjalankan tiga langkah:

1. **Membaca konfigurasi** model akhir dari folder hasil Pertemuan 7 di Google Drive.
2. **Menyiapkan model.** Jika model akhir sudah tersimpan di Google Drive, model langsung dimuat (sekitar 1–2 menit). Jika belum, model dilatih satu kali pada seluruh data training (sekitar 50 menit) lalu disimpan, sehingga pemakaian berikutnya tidak perlu melatih ulang.
3. **Mengklasifikasikan email** yang ditempel pada kotak isian.

Model dilatih hanya pada data training, sama seperti model yang diuji pada Evaluasi Akhir. Data testing tidak dipakai.

# A. Konfigurasi Model Akhir

Sel berikut menghubungkan Google Drive dan membaca `final_config.json` serta prediksi *out-of-fold* dari folder hasil Pertemuan 7. Prediksi *out-of-fold* dipakai untuk melatih *meta-model*. Jika Google Drive gagal terhubung atau hasil Pertemuan 7 belum ada, notebook berhenti di sel ini.

In [ ]:
import sys
import subprocess
import os
import re
import time
import gc
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')  # tanpa try/except: kegagalan menghentikan notebook di sini
    DRIVE_ROOT = '/content/drive/MyDrive/Group1_ML'
else:
    DRIVE_ROOT = '.'  # hanya untuk menjalankan notebook di luar Colab
M07_DIR = f'{DRIVE_ROOT}/m07_artifacts'
MODEL_DIR = f'{DRIVE_ROOT}/final_model'
os.makedirs(MODEL_DIR, exist_ok=True)
with open(os.path.join(MODEL_DIR, '_write_test.txt'), 'w') as handle:
    handle.write('ok')  # menulis ke folder model sekarang, sebelum pelatihan dimulai

with open(f'{M07_DIR}/final_config.json') as handle:
    config = json.load(handle)

LABEL_NAMES = {0: 'Ham', 1: 'Phish', 2: 'Spam'}
CLASS_IDS = [0, 1, 2]
FINAL_NAME = config['final_model']
BASE_NAMES = config['final_model_bases']
BASE_MODELS = [name.split()[0] for name in BASE_NAMES]  # 'MNB', 'LR', 'LightGBM'
HAM_BIAS = config['ham_bias_b']
RANDOM_STATE = config['random_state']

PUNCTUATION_PATTERN = config['punctuation_pattern']
TIMESTAMP_PATTERN = config['timestamp_pattern']
LINK_PATTERN = config['link_pattern']
WEB_PARTS_PATTERN = config['web_parts_pattern']
NUMBER_PATTERN = config['number_pattern']
DATE_PATTERN = config['date_pattern']
STOPWORD_PATTERN = r'\b(?:' + '|'.join(sorted(config['stopwords'], key=len, reverse=True)) + r')\b'


def normalize_v3(texts):
    return (texts.str.lower()
            .str.replace(PUNCTUATION_PATTERN, ' ', regex=True)
            .str.replace(TIMESTAMP_PATTERN, ' ', regex=True)
            .str.replace(LINK_PATTERN, ' urltoken ', regex=True)
            .str.replace(WEB_PARTS_PATTERN, ' ', regex=True)
            .str.replace(NUMBER_PATTERN, ' numtoken ', regex=True)
            .str.replace(DATE_PATTERN, ' datetoken ', regex=True)
            .str.replace(STOPWORD_PATTERN, ' ', regex=True))


print('Folder model:', os.path.abspath(MODEL_DIR))
display(Markdown(f'Model akhir: **{FINAL_NAME}** (model dasar: {", ".join(BASE_NAMES)}), threshold Ham b = {HAM_BIAS:.3f}.'))

# B. Model Akhir

Sel berikut menyiapkan setiap komponen model akhir:

| Komponen | Berkas di `final_model/` | Jika belum ada |
| --- | --- | --- |
| Count unigram + bigram | `count_vectorizer.joblib` | Dibentuk dari data training V3 |
| Multinomial Naive Bayes | `MNB.joblib` | Dilatih pada count unigram + bigram |
| LightGBM | `LightGBM.joblib` | Dilatih pada count unigram (`min_df` dari konfigurasi) |
| TF-IDF + Logistic Regression | `LR.joblib` | Dilatih pada TF-IDF unigram |
| *Meta-model* | `meta.joblib` | Dilatih pada prediksi *out-of-fold* Pertemuan 7 |

Komponen yang sudah tersimpan langsung dimuat. Komponen yang belum ada dilatih dengan konfigurasi Pertemuan 7, lalu langsung disimpan. Data training hanya dimuat jika ada model dasar yang perlu dilatih.

In [ ]:
def ensure(package, module=None):
    try:
        __import__(module or package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])


ensure('lightgbm')
import joblib
import lightgbm as lgb
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
PATHS = {name: f'{MODEL_DIR}/{name}.joblib' for name in ['count_vectorizer', 'MNB', 'LightGBM', 'LR', 'meta']}
needs_data = [name for name in ['count_vectorizer', 'MNB', 'LightGBM', 'LR'] if not os.path.exists(PATHS[name])]

if needs_data:
    ensure('datasets')
    from datasets import load_dataset
    from sklearn.model_selection import train_test_split

    print('Melatih komponen:', needs_data)
    raw_df = load_dataset('locuoco/the-biggest-spam-ham-phish-email-dataset-300000', split='train').to_pandas()
    data = raw_df[['text', 'label']].dropna(subset=['text', 'label']).copy()
    data = data[data['label'].isin(CLASS_IDS)].copy()
    data['label'] = data['label'].astype(int)
    data['clean_text'] = data['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    data = data[data['clean_text'].ne('')].copy()
    data['dedup_key'] = data['clean_text'].str.lower()
    labels_per_key = data.groupby('dedup_key')['label'].nunique()
    data = data.loc[~data['dedup_key'].isin(labels_per_key[labels_per_key.gt(1)].index)]
    df_clean = data.drop_duplicates(subset=['dedup_key'], keep='first')[['clean_text', 'label']].reset_index(drop=True)
    del raw_df, data
    X_train, _, y_train, _ = train_test_split(df_clean['clean_text'], df_clean['label'], test_size=0.20,
                                              random_state=RANDOM_STATE, stratify=df_clean['label'])
    y_train = y_train.to_numpy()
    del df_clean
    V3_train = normalize_v3(X_train.reset_index(drop=True))
    del X_train
    _ = gc.collect()
    print(f'Data training: {len(V3_train):,} pesan')

    if any(name in needs_data for name in ['count_vectorizer', 'MNB', 'LightGBM']):
        start = time.perf_counter()
        if os.path.exists(PATHS['count_vectorizer']):
            count_vectorizer = joblib.load(PATHS['count_vectorizer'])
            X_count = count_vectorizer.transform(V3_train)
        else:
            count_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, dtype=np.float32)
            X_count = count_vectorizer.fit_transform(V3_train)
            joblib.dump(count_vectorizer, PATHS['count_vectorizer'])
        print(f'Count unigram + bigram: {X_count.shape[1]:,} fitur, {(time.perf_counter() - start) / 60:.1f} menit, tersimpan')

        if 'MNB' in needs_data:
            start = time.perf_counter()
            steps = [('classifier', MultinomialNB(alpha=config['mnb']['alpha']))]
            if config['mnb']['chi2_k']:
                steps.insert(0, ('chi2', SelectKBest(chi2, k=config['mnb']['chi2_k'])))
            joblib.dump(Pipeline(steps).fit(X_count, y_train), PATHS['MNB'])
            print(f'MNB: {(time.perf_counter() - start) / 60:.1f} menit, tersimpan')

        if 'LightGBM' in needs_data:
            start = time.perf_counter()
            min_df = int(re.search(r'min_df=(\d+)', config['lightgbm']['representation']).group(1))
            unigram = np.array([' ' not in term for term in count_vectorizer.get_feature_names_out()])
            document_frequency = np.asarray((X_count > 0).sum(axis=0)).ravel()
            columns = np.flatnonzero(unigram & (document_frequency >= min_df))
            params = {key: value for key, value in config['lightgbm'].items() if key != 'representation'}
            booster = lgb.LGBMClassifier(**params, verbose=-1, n_jobs=-1).fit(X_count[:, columns], y_train)
            joblib.dump({'model': booster, 'columns': columns}, PATHS['LightGBM'])
            print(f'LightGBM: {(time.perf_counter() - start) / 60:.1f} menit, tersimpan')
        del X_count
        _ = gc.collect()

    if 'LR' in needs_data:
        start = time.perf_counter()
        lr_pipeline = make_pipeline(
            TfidfVectorizer(lowercase=True, ngram_range=(1, 1), sublinear_tf=True, dtype=np.float32),
            LogisticRegression(C=config['lr']['C'], max_iter=1000, random_state=RANDOM_STATE),
        ).fit(V3_train, y_train)
        joblib.dump(lr_pipeline, PATHS['LR'])
        print(f'LR: {(time.perf_counter() - start) / 60:.1f} menit, tersimpan')
    del V3_train
    _ = gc.collect()

if not os.path.exists(PATHS['meta']):
    m07_oof = np.load(f'{M07_DIR}/oof_log_proba.npz')
    features = np.hstack([np.clip(m07_oof[name.replace(' ', '_')], -30, 0) for name in BASE_NAMES])
    meta = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    joblib.dump(meta.fit(features, m07_oof['y_train']), PATHS['meta'])
    print('Meta-model: tersimpan')

start = time.perf_counter()
models = {name: joblib.load(path) for name, path in PATHS.items()}
print(f'Semua komponen dimuat dalam {time.perf_counter() - start:.0f} detik:', sorted(os.listdir(MODEL_DIR)))

# C. Fungsi Klasifikasi

Sel berikut mendefinisikan fungsi `classify`. Untuk setiap email, fungsi ini menjalankan langkah yang sama dengan model akhir:

1. Spasi dirapikan, lalu teks diseragamkan menjadi teks V3.
2. Setiap model dasar menghitung probabilitas ketiga kelas.
3. *Meta-model* menggabungkan probabilitas tersebut.
4. Nilai b ditambahkan pada log-probabilitas Ham, lalu kelas dengan skor tertinggi dipilih.

Hasilnya ditampilkan sebagai tindakan pada kotak masuk, probabilitas dari setiap model, dan kata yang paling mendorong keputusan Logistic Regression.

| Prediksi | Tindakan |
| --- | --- |
| Ham | Masuk kotak masuk |
| Spam | Dipindahkan ke folder spam |
| Phish | Dipindahkan ke folder spam dengan peringatan phishing |

In [ ]:
ACTIONS = {0: 'masuk kotak masuk', 1: 'dipindahkan ke folder spam dengan peringatan phishing', 2: 'dipindahkan ke folder spam'}
lr_vectorizer, lr_model = models['LR'][0], models['LR'][-1]
lr_terms = lr_vectorizer.get_feature_names_out()


def base_log_proba(v3_text):
    X_count = models['count_vectorizer'].transform(v3_text)
    lgbm = models['LightGBM']
    return {
        'MNB': models['MNB'].predict_log_proba(X_count),
        'LR': models['LR'].predict_log_proba(v3_text),
        'LightGBM': np.log(np.clip(lgbm['model'].predict_proba(X_count[:, lgbm['columns']]), 1e-15, 1.0)),
    }


def classify(text, n_words=10):
    v3_text = normalize_v3(pd.Series([' '.join(str(text).split())]))
    words = v3_text.iloc[0].split()
    if not words:
        display(Markdown('Teks kosong setelah penyeragaman. Tempel isi email yang memuat kata.'))
        return None
    base = base_log_proba(v3_text)
    meta_log_proba = models['meta'].predict_log_proba(np.hstack([np.clip(base[m], -30, 0) for m in BASE_MODELS]))
    adjusted = meta_log_proba.copy()
    adjusted[:, 0] += HAM_BIAS
    predicted = int(adjusted.argmax(axis=1)[0])

    display(Markdown(f'### Hasil: **{LABEL_NAMES[predicted]}** (email {ACTIONS[predicted]})'))
    rows = {name: np.exp(base[model][0]) for name, model in zip(BASE_NAMES, BASE_MODELS)}
    rows[f'{FINAL_NAME} (sebelum threshold)'] = np.exp(meta_log_proba[0])
    display(pd.DataFrame(rows, index=[f'P({LABEL_NAMES[c]})' for c in CLASS_IDS]).T.round(3))
    ratio = np.exp(-HAM_BIAS)
    display(Markdown(
        f'Dengan b = {HAM_BIAS:.3f}, email dianggap Ham jika P(Ham) dari *meta-model* minimal {ratio:.3f} kali '
        f'probabilitas kelas lain yang terbesar. Pada email ini, P(Ham) = {np.exp(meta_log_proba[0, 0]):.3f} dan '
        f'probabilitas kelas lain terbesar = {np.exp(meta_log_proba[0, 1:]).max():.3f}.'
    ))

    X_tfidf = lr_vectorizer.transform(v3_text)
    row = list(lr_model.classes_).index(predicted)
    contribution = X_tfidf.multiply(lr_model.coef_[row]).tocsr()
    positive = np.flatnonzero(contribution.data > 0)
    order = positive[np.argsort(-contribution.data[positive])][:n_words]
    known = X_tfidf.nnz / max(len(set(words)), 1)
    notes = [f'{known * 100:.0f}% kata unik pada email ini dikenali oleh Logistic Regression.']
    if len(words) <= 10:
        notes.append('Email ini memuat 10 kata atau kurang setelah penyeragaman. Pada evaluasi, kesalahan paling sering terjadi pada pesan sependek ini.')
    if known < 0.5:
        notes.append('Kurang dari separuh kata dikenali model, misalnya karena email tidak berbahasa Inggris, sehingga hasil ini kurang dapat diandalkan.')
    display(Markdown(f'**Kata yang paling mendorong prediksi {LABEL_NAMES[predicted]} pada Logistic Regression**'))
    if len(order):
        display(pd.DataFrame({
            'kata': lr_terms[contribution.indices[order]],
            'kontribusi (TF-IDF × bobot)': contribution.data[order].round(3),
        }))
    else:
        display(Markdown(f'Tidak ada kata pada email ini yang mendorong Logistic Regression ke arah {LABEL_NAMES[predicted]}; keputusan berasal dari model dasar lain dan threshold Ham.'))
    display(Markdown('**Teks V3 (teks yang dibaca model):** `' + ' '.join(words)[:600] + '`\n\n' + '\n'.join(f'- {n}' for n in notes)))
    return LABEL_NAMES[predicted]

# D. Contoh Email

Sel berikut mengklasifikasikan tiga email contoh yang ditulis sendiri untuk demo ini: satu email kerja, satu email bergaya phishing, dan satu iklan obat.

In [ ]:
EXAMPLES = {
    'Email kerja': (
        'Hi team, the project meeting is moved to Thursday at 3 pm in room 204. Please bring the updated budget '
        'figures and your comments on the draft report. Let me know if the new time does not work for you. Thanks, Maria'
    ),
    'Email bergaya phishing': (
        'Dear customer, we detected unusual sign-in activity on your account. Your account will be suspended within '
        '24 hours unless you verify your identity. Click the link below to confirm your details: '
        'http://secure-account-check.example.com/login Thank you, Customer Security Team'
    ),
    'Iklan obat': (
        'Lowest prices on all medications! Save up to 80% today. Free worldwide shipping, no prescription needed, '
        'discreet packaging. Visit our online pharmacy now: http://www.best-meds-online.example.net'
    ),
}
for title, text in EXAMPLES.items():
    display(Markdown(f'---\n## {title}\n\n> {text}'))
    classify(text)

# E. Klasifikasi Email Sendiri

Tempel isi email berbahasa Inggris pada kotak berikut, lalu tekan **Klasifikasi**. Subjek email boleh ikut ditempel. Kotak dapat dipakai berulang kali tanpa menjalankan ulang sel.

In [ ]:
import ipywidgets as widgets

email_box = widgets.Textarea(placeholder='Tempel isi email di sini...', layout=widgets.Layout(width='100%', height='220px'))
classify_button = widgets.Button(description='Klasifikasi', button_style='primary')
clear_button = widgets.Button(description='Kosongkan')
result_area = widgets.Output()


def on_classify(_):
    result_area.clear_output()
    with result_area:
        classify(email_box.value)


def on_clear(_):
    email_box.value = ''
    result_area.clear_output()


classify_button.on_click(on_classify)
clear_button.on_click(on_clear)
display(email_box, widgets.HBox([classify_button, clear_button]), result_area)

Jika kotak isian di atas tidak tampil, tempel email di antara tanda kutip tiga pada sel berikut, lalu jalankan sel tersebut.

In [ ]:
EMAIL_TEXT = """

"""
if EMAIL_TEXT.strip():
    classify(EMAIL_TEXT)
else:
    print('EMAIL_TEXT masih kosong. Tempel isi email di antara tanda kutip tiga, lalu jalankan sel ini.')

# F. Batasan Pemakaian

Hasil demo ini perlu dibaca dengan batasan model yang ditemukan selama proyek:

1. **Bahasa.** Model dilatih pada email berbahasa Inggris. Email dalam bahasa lain sebagian besar katanya tidak dikenali.
2. **Sumber data.** Setiap kelas berasal dari kumpulan email yang berbeda, dan model masih mengenali sebagian ciri sumbernya (misalnya nama karyawan Enron pada Ham). Email dari organisasi lain dapat diklasifikasikan kurang tepat.
3. **Pesan pendek.** Kesalahan paling sering terjadi pada pesan dengan 10 kata atau kurang.
4. **Phish dan Spam.** Sekitar sepertiga Phish terklasifikasi sebagai Spam pada evaluasi. Email tersebut tetap tersaring dari kotak masuk, tetapi tanpa peringatan phishing.
5. **Threshold Ham.** Nilai b membuat model lebih berhati-hati sebelum membuang email: email yang ragu-ragu cenderung dimasukkan ke kotak masuk.